In [1]:
from pathlib import Path

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

import pandas as pd

triplets = pd.read_csv(
    DATA_DIR / "train_triplets.txt",
    sep="\t",
    dtype={"user_id": "category", "song_id": "category", "play_count": "uint16"},
    header=None,
    names=["user_id", "song_id", "play_count"]
)

print(triplets.shape)

print("Users :", triplets["user_id"].nunique())
print("Songs :", triplets["song_id"].nunique())

print("Duplicate rows :", triplets.duplicated().sum())

(48373586, 3)
Users : 1019318
Songs : 384546
Duplicate rows : 0


In [2]:
genres = pd.read_csv(
    DATA_DIR / "p02_msd_tagtraum_cd2.cls",
    sep="\t",
    comment="#",
    header=None,
    names=["track_id", "major_genre", "minor_genre"]
)

genres.head()

,track_id,major_genre,minor_genre
0,TRAAAAK128F9318786,Rock,NaN
1,TRAAAAW128F429D538,Rap,NaN
2,TRAAABD128F429CF47,Rock,RnB
3,TRAAADJ128F4287B47,Rock,NaN
4,TRAAADZ128F9348C2E,Latin,NaN


In [3]:
tracks = pd.read_csv(
    DATA_DIR / "p02_unique_tracks.txt",
    sep="<SEP>",
    engine="python",
    header=None,
    names=["track_id", "song_id", "artist", "title"]
)


In [4]:
print("listening rows:", triplets.shape)
print("track rows:", tracks.shape)
print("genre rows:", genres.shape)

# Keep the 48M-row listening table narrow. Join metadata only after
# aggregating or selecting a small set of songs.
metadata = (
    tracks.merge(genres, on="track_id", how="left")
    .drop_duplicates("song_id")
    [["track_id", "song_id", "artist", "title", "major_genre", "minor_genre"]]
    .reset_index(drop=True)
)
songs = triplets
metadata.head()

listening rows: (48373586, 3)
track rows: (1000000, 4)
genre rows: (280831, 3)


,track_id,song_id,artist,title,major_genre,minor_genre
0,TRMMMYQ128F932D901,SOQMMHC12AB0180CB8,Faster Pussy cat,Silent Night,NaN,NaN
1,TRMMMKD128F425225D,SOVFVAK12A8C1350D9,Karkkiautomaatti,Tanssi vaan,NaN,NaN
2,TRMMMRX128F93187D9,SOGTUKN12AB017F4F1,Hudson Mohawke,No One Could Ever,Electronic,Rap
3,TRMMMCH128F425532C,SOBNYVR12A8C13558C,Yerba Brava,Si Vos Querés,NaN,NaN
4,TRMMMWA128F426B589,SOHSBXH12A8C13B0DF,Der Mystic,Tangle Of Aspens,NaN,NaN


In [5]:
import sys
MODULE_DIR = Path("notebooks") if Path("notebooks/Myspotify.py").exists() else Path(".")
sys.path.insert(0, str(MODULE_DIR.resolve()))

from importlib import reload
import Myspotify

reload(Myspotify)

from Myspotify import MySpotify

myspotify = MySpotify(songs, metadata=metadata)
top250 = myspotify.top_250()
top250.to_csv(DATA_DIR / "Top_250_track.csv")
print("--------------- TOP 250 TRACK SONGS ---------------")
print(top250)



--------------- TOP 250 TRACK SONGS ---------------
                                                artist  \
1                                        Dwight Yoakam   
2                                                Björk   
3                                        Kings Of Leon   
4                                             Harmonia   
5    Barry Tuckwell/Academy of St Martin-in-the-Fie...   
..                                                 ...   
246                                   Triple Six Mafia   
247                         The Red Jumpsuit Apparatus   
248                                        Linkin Park   
249                           Selena Gomez & The Scene   
250                       Creedence Clearwater Revival   

                                                 title  total_play_count  
1                                       You're The One            726885  
2                                                 Undo            648239  
3                         

In [6]:
required_genres = MySpotify.REQUIRED_GENRES
genre_rankings = {genre: myspotify.top_100_by_genre(genre) for genre in required_genres}
{genre: len(ranking) for genre, ranking in genre_rankings.items()}

,artist,title,total_play_count
1,Metallica,Master Of Puppets,98667
2,Five Finger Death Punch,Bad Company,42348
3,Nevermore,Matricide,33607
4,Metallica,The Unforgiven III,22048
5,Apocalyptica,Refuse / Resist,17099
...,...,...,...
96,Job For A Cowboy,Martyrdom Unsealed,3499
97,Danzig,It's Coming Down,3484
98,Metallica,The Ecstasy Of Gold,3425
99,Parkway Drive,Carrion,3399


In [7]:
# • Collections: 50 songs about love, 50 songs about war, 50 songs about happiness,
#  50 songs about loneliness, 50 songs about money. Content-based approach.

lyrics_df, vocabulary = myspotify.load_mxm_dataset(DATA_DIR / "mxm_dataset_train.txt")

lyrics_df.head()

,track_id,mxm_track_id,word_counts
0,TRAAAAV128F421A322,4623710,"{'i': 6, 'the': 4, 'you': 2, 'to': 2, 'and': 5..."
1,TRAAABD128F429CF47,6477168,"{'i': 10, 'you': 17, 'to': 8, 'and': 2, 'a': 2..."
2,TRAAAED128E0783FAB,2516445,"{'i': 28, 'the': 15, 'you': 2, 'to': 12, 'and'..."
3,TRAAAEF128F4273421,3759847,"{'i': 5, 'the': 4, 'you': 3, 'to': 2, 'and': 1..."
4,TRAAAEW128F42930C0,3783760,"{'i': 4, 'to': 5, 'and': 7, 'a': 2, 'me': 4, '..."


In [8]:
spotify = MySpotify(songs, lyrics_df, metadata=metadata, vocabulary=vocabulary)
collection_results = {
    keyword: spotify.collections(keyword)
    for keyword in ("love", "war", "happiness", "loneliness", "money")
}
{keyword: len(result) for keyword, result in collection_results.items()}

,artist,title,keyword_count,total_play_count
1,Dilated Peoples,Trade Money,101,108
2,50 Cent,I Get Money,82,6140
3,Clipse,Dirty Money,79,30
4,Hot Action Cop,Doom Boom(album/edited version),71,42
5,Zion,Money,59,2
6,Gorilla Zoe,Money Man (Explicit Album Version),51,252
7,David Wilcox,Blood Money,49,26
8,Extreme,Money (In God We Trust),42,162
9,Girls Aloud,Money,38,91
10,Rick Ross / EbonyLove,Money Make Me Come,37,871


In [9]:
import sklearn


In [10]:
spotify.train_test_split()

Train: (38698868, 3)
Test : (9674718, 3)


In [11]:
spotify.build_user_item_matrix()

(1019318, 384546)


In [12]:
spotify.fit_user_model()
spotify.fit_item_model()
print("User model trained successfully.")

User model trained successfully.


In [13]:
user = songs["user_id"].iloc[0]

spotify.similar_users(user)

(array(['83c34e209385675c977f8b7c0cbab7ffd60a0822',
        'a6472b1f6721e22496d0bcfcc90faa83b432f03b',
        'bdb06f88436d3e49160d185c00fd19b150386345',
        'b774c403bc03924d61bf96dca0a44392e4d1a3f6',
        '7b96739fce312cdded3cbbf0a0729a2d0c4c0b93',
        'd4ddec202eef5de3e681a923f1f133b9226714f1',
        '7113897a636503b447a91b187dea5c6c7a6b06dc',
        'a272f50254261d37a61385f3abfbede24e1d256b',
        '6b22a8f822f185ac8e968d2fa45755f49b54d8a2',
        'c7afe07b86190e29ab5bd53c79ab251b3708cf26'], dtype=object),
 array([0.41604555, 0.38233602, 0.38187957, 0.37775028, 0.37280595,
        0.37242675, 0.37151754, 0.37143964, 0.36963803, 0.3692485 ],
       dtype=float32))

In [14]:
user = songs["user_id"].iloc[0]

spotify.recommend_for_user(user)

,song_id,artist,title,score
0,SOIAIVS12A8C137973,Anne Dudley,Turbulent Serenity (Instrumental),11.888561
1,SOSGIVA12A58A7DA64,Beach House,Heart of Chambers,4.086693
2,SOYZLOI12AB0180146,Nightmares On Wax,You Wish,3.343658
3,SOHALKJ12AB017FCC2,Godsmack,Speak,2.676352
4,SOOQOEJ12AB0183C50,The Quantic Soul Orchestra,Terrapin,2.266502
5,SOKAZAY12A8C13F3A7,JT Tha Bigga Figga,Truth Rap (Featuring Sean T_ The Game),1.911680
6,SOMKLND12A81C207B2,The Game,Troublesome,1.911680
7,SOKOSPK12A8C13C088,Tegan And Sara,Call It Off (Album Version),1.911680
8,SOYWRLV12AB0186090,Panic At The Disco,I Write Sins Not Tragedies [Live In Chicago],1.664182
9,SONBLCG12A8AE46347,Jamiroquai,Virtual Insanity,1.511001


In [15]:
user_precision = spotify.precision_at_k(sample_size=10)
item_precision = spotify.precision_at_k_for_tracks(sample_size=10)
sample_track = songs["song_id"].iloc[0]
track_recommendations = spotify.recommend_for_track(sample_track)
track_recommendations

Precision@10: 0.0200


np.float64(0.02)